In [3]:
!pip install transformers

In [4]:
!pip install gradio -q

In [5]:
# --- Install dependencies ---
!pip install gradio diffusers transformers accelerate safetensors openai-whisper -q

import gradio as gr
import torch
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline
import whisper
from PIL import Image

# --- Device setup (auto-detect GPU/CPU) ---
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# --- Load Models ---
# Photorealistic model for better product shots
text2img_pipe = StableDiffusionPipeline.from_pretrained(
    "dreamlike-art/dreamlike-photoreal-2.0",
    torch_dtype=dtype
).to(device)

img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "dreamlike-art/dreamlike-photoreal-2.0",
    torch_dtype=dtype
).to(device)

whisper_model = whisper.load_model("base")

# --- Functions ---
def generate_image_from_text(prompt):
    if not prompt:
        return None
    image = text2img_pipe(prompt, guidance_scale=9).images[0]
    return image

def generate_image_from_speech(audio_file):
    if audio_file is None:
        return None
    result = whisper_model.transcribe(audio_file)
    text_prompt = result["text"]
    image = text2img_pipe(text_prompt, guidance_scale=9).images[0]
    return image, text_prompt

def generate_product_image(prompt, input_image, strength=0.45, guidance_scale=10, seed=42):
    if input_image is None or not prompt:
        return None
    init_image = Image.open(input_image).convert("RGB").resize((512, 512))
    generator = torch.manual_seed(seed)
    image = img2img_pipe(
        prompt=prompt,
        image=init_image,
        strength=strength,
        guidance_scale=guidance_scale,
        generator=generator
    ).images[0]
    return image

# --- Gradio UI ---
with gr.Blocks() as demo:
    gr.Markdown("## 🖌️ Vizzy Chat — Text, Speech & Image-to-Image")

    with gr.Tab("Text Prompt"):
        text_in = gr.Textbox(placeholder="Enter a text prompt...")
        text_out = gr.Image(type="pil")
        text_in.submit(generate_image_from_text, text_in, text_out)

    with gr.Tab("Speech Prompt"):
        audio_in = gr.Audio(type="filepath", label="Upload or record audio")
        audio_out_img = gr.Image(type="pil")
        audio_out_text = gr.Textbox(label="Transcribed Prompt")
        audio_in.change(generate_image_from_speech, audio_in, [audio_out_img, audio_out_text])

    with gr.Tab("Image Prompt"):
        img_in = gr.Image(type="filepath", label="Upload an image")
        img_prompt = gr.Textbox(placeholder="Describe how to transform this image...")
        img_out = gr.Image(type="pil")
        img_prompt.submit(generate_product_image, [img_prompt, img_in], img_out)

demo.launch()


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--dreamlike-art--dreamlike-photoreal-2.0/snapshots/d9e27ac81cfa72def39d74ca673219c349f0a0d5/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--dreamlike-art--dreamlike-photoreal-2.0/snapshots/d9e27ac81cfa72def39d74ca673219c349f0a0d5/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://360adac59a128f8001.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
